# Parallel merge (chapter 13)

Based on chapter 13 of *Programming Massively Parallel Processors* (Hwu, Kirk, El Hajj). The earlier patterns of the course (map, stencil, histogram, reduction, scan) have one thing in common: thread `t` knows from its index alone which input elements it must read. Merge breaks that rule. Where thread `t` should start reading depends on the **values** in the data, so each thread first has to *search* for its input. The tool for that search, the **co-rank function**, is the main idea of the chapter, and chapter 14 uses it again to build a parallel merge sort.

This notebook goes **step by step**: 5 small programs, each adding one idea to the previous one, from the simplest kernel up. The inputs are tiny and printed, so you can follow the algorithm by hand.

The full theory, the exercises and their solutions are in `ec2/11_merge/README.md`. The slides of the session are the companion of this notebook.

### How this notebook works

Every program comes as a pair of cells:

1. a cell that starts with `%%writefile name.cu`: running it **saves** the CUDA program as a file on the
   Colab machine (it does not run anything yet);
2. a cell that starts with `!nvcc`: it **compiles** that file with NVIDIA's compiler and **runs** it on
   the GPU. The `!` sends the line to the machine's shell.

Every program computes its result on the GPU **and** with a simple loop on the CPU, compares the two and
prints **PASS** when they agree (**FAIL** otherwise). Read the code cell before running it, then read
the output: the Markdown after each run says what to look at.

You can edit any `%%writefile` cell (change a size, a constant, a kernel) and run the pair again.

## Setup: get a GPU

A new Colab notebook runs on a machine **without a GPU**. "Enabling the GPU" is not something the code
does: it means asking Colab for a machine that has an NVIDIA GPU, with the NVIDIA driver and the CUDA
toolkit (the `nvcc` compiler) already installed.

**Do this once, before running anything:** *Runtime → Change runtime type → Hardware accelerator: T4 GPU
→ Save*. Colab restarts the machine; files written before the change are lost, so run the notebook from
the top afterwards.

The next cell checks it worked. `nvidia-smi` lists the GPU (usually a *Tesla T4*) and `nvcc --version`
prints the CUDA compiler version. If you see *command not found* or *No devices were found*, the runtime
has no GPU: change it as above.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

## The `cuda_compat.h` header: what it is and when you need it

Every program in this course starts with `#include "cuda_compat.h"`. The header exists so that **the
same `.cu` file runs everywhere**, with or without an NVIDIA GPU:

| Where you run | Compiler | What `cuda_compat.h` does |
|---|---|---|
| Google Colab with a T4 GPU (this notebook) | `nvcc` | Almost nothing: it includes the real CUDA runtime, so the code runs on the real GPU. |
| Your own computer **with** an NVIDIA GPU | `nvcc` | Exactly the same as on Colab. |
| A laptop **without** an NVIDIA GPU (any Mac, most laptops) | `g++` or `clang++` | It **emulates** CUDA on the CPU: each block's threads are real CPU threads, `__syncthreads()` is a barrier, `__shared__` memory is shared, atomics and warp shuffles work. Results are correct; times are meaningless. |

It also gives three small helpers that the programs use with `nvcc` too:

- `launch(kernel, grid, block, args...)`: with `nvcc` it is **exactly** `kernel<<<grid, block>>>(args...)`.
  The `<<<...>>>` syntax only exists in `nvcc`, so a file that used it could not be compiled with `g++`;
  `launch` keeps one source for both.
- `CHECK(call)`: stops with the error message, file and line if a CUDA call fails (introduced in
  `01_first_kernels`, step 8).
- `cdiv(a, b)`: the ceiling division used to compute how many blocks a grid needs.

**On a computer with an NVIDIA GPU there is nothing to enable.** Install the NVIDIA driver and the CUDA
Toolkit (`nvcc`), then compile exactly as the cells below do:
`nvcc -O2 -arch=native -I../common file.cu -o file && ./file` (`-arch=native` compiles for the GPU in
that machine). You still include `cuda_compat.h`, because the programs call `launch` and `CHECK`; under
`nvcc` it does not emulate anything. If you prefer plain CUDA, write `kernel<<<grid, block>>>(args...)`
instead of `launch(...)` and the header is no longer needed for the launch.

**Without an NVIDIA GPU**, compile the same file with
`g++ -std=c++17 -O2 -pthread -x c++ -I../common file.cu -o file && ./file`.

The next cell saves the header on the Colab machine. You do not need to read it: just run it.

In [ ]:
%%writefile cuda_compat.h
#pragma once

#include <chrono>
#include <cstdio>
#include <cstdlib>

#ifdef __CUDACC__

#include <cuda_runtime.h>
#include <cuda/atomic>

#define CUDA_BACKEND "GPU (nvcc)"
#define BY_BACKEND(gpu, cpu) (gpu)

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  kernel<<<grid, block>>>(args...);
}

#else

#include <atomic>
#include <cstring>
#include <memory>
#include <thread>
#include <type_traits>
#include <vector>

#define CUDA_BACKEND "CPU emulation (threads)"
#define BY_BACKEND(gpu, cpu) (cpu)

#define __global__
#define __device__
#define __host__
#define __constant__
#define __shared__ static
#define __restrict__

struct dim3 {
  unsigned x;
  unsigned y;
  unsigned z;

  dim3(unsigned x_ = 1, unsigned y_ = 1, unsigned z_ = 1) : x(x_), y(y_), z(z_) {
  }
};

struct uint3 {
  unsigned x;
  unsigned y;
  unsigned z;
};

struct float4 {
  float x;
  float y;
  float z;
  float w;
};

inline thread_local uint3 threadIdx{0, 0, 0};
inline thread_local uint3 blockIdx{0, 0, 0};
inline dim3 blockDim;
inline dim3 gridDim;

constexpr int warpSize = 32;

namespace emu {

constexpr unsigned kWarpSize = warpSize;
constexpr unsigned kFullWarpMask = 0xffffffffu;
constexpr int kCudaAtomicFunctionOrder = __ATOMIC_RELAXED;

class Barrier {
 public:
  explicit Barrier(unsigned threadCount) : threadCount_(threadCount) {
  }

  void arriveAndWait() {
    const unsigned generationOnArrival = generation_.load(std::memory_order_acquire);
    const bool lastToArrive = arrived_.fetch_add(1, std::memory_order_acq_rel) + 1 == threadCount_;

    if (lastToArrive) {
      releaseEveryWaitingThread();
    } else {
      yieldUntilGenerationEnds(generationOnArrival);
    }
  }

 private:
  void releaseEveryWaitingThread() {
    arrived_.store(0, std::memory_order_relaxed);
    generation_.fetch_add(1, std::memory_order_acq_rel);
  }

  void yieldUntilGenerationEnds(unsigned generation) const {
    while (generation_.load(std::memory_order_acquire) == generation) {
      std::this_thread::yield();
    }
  }

  std::atomic<unsigned> arrived_{0};
  std::atomic<unsigned> generation_{0};
  const unsigned threadCount_;
};

inline Barrier* blockBarrier = nullptr;

using LaneBits = unsigned long long;

class Warp {
 public:
  explicit Warp(unsigned laneCount) : laneCount_(laneCount), barrier_(laneCount), published_{} {
  }

  unsigned laneCount() const {
    return laneCount_;
  }

  unsigned everyLaneMask() const {
    if (laneCount_ == kWarpSize) {
      return kFullWarpMask;
    }
    return (1u << laneCount_) - 1u;
  }

  bool hasLane(int lane) const {
    return lane >= 0 && lane < static_cast<int>(laneCount_);
  }

  void publish(unsigned lane, LaneBits bits) {
    __atomic_store_n(&published_[lane], bits, __ATOMIC_RELAXED);
  }

  LaneBits publishedBy(int lane) const {
    return __atomic_load_n(&published_[lane], __ATOMIC_RELAXED);
  }

  void waitForEveryLane() {
    barrier_.arriveAndWait();
  }

 private:
  const unsigned laneCount_;
  Barrier barrier_;
  LaneBits published_[kWarpSize];
};

inline thread_local Warp* warpOfThisThread = nullptr;
inline thread_local unsigned laneOfThisThread = 0;

template <class T>
LaneBits toLaneBits(T value) {
  static_assert(sizeof(T) <= sizeof(LaneBits), "shuffle of a type wider than 8 bytes");
  LaneBits bits = 0;
  std::memcpy(&bits, &value, sizeof(T));
  return bits;
}

template <class T>
T fromLaneBits(LaneBits bits) {
  T value;
  std::memcpy(&value, &bits, sizeof(T));
  return value;
}

template <class T>
T readValueOfLane(T myValue, int sourceLane) {
  Warp& myWarp = *warpOfThisThread;
  myWarp.publish(laneOfThisThread, toLaneBits(myValue));
  myWarp.waitForEveryLane();

  T received = myValue;
  if (myWarp.hasLane(sourceLane)) {
    received = fromLaneBits<T>(myWarp.publishedBy(sourceLane));
  }

  myWarp.waitForEveryLane();
  return received;
}

inline int positionInGroup(int lane, int width) {
  return lane % width;
}

inline int firstLaneOfGroup(int lane, int width) {
  return lane / width * width;
}

inline bool inSameGroup(int lane, int otherLane, int width) {
  return lane / width == otherLane / width;
}

inline int shflDownSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) + delta < width;
  return sourceInGroup ? lane + delta : lane;
}

inline int shflUpSourceLane(int lane, int delta, int width) {
  const bool sourceInGroup = positionInGroup(lane, width) >= delta;
  return sourceInGroup ? lane - delta : lane;
}

inline int shflXorSourceLane(int lane, int laneMask, int width) {
  const int partner = lane ^ laneMask;
  return inSameGroup(lane, partner, width) ? partner : lane;
}

inline int shflSourceLane(int lane, int srcLane, int width) {
  const int wrappedSrcLane = (srcLane % width + width) % width;
  return firstLaneOfGroup(lane, width) + wrappedSrcLane;
}

inline int myLane() {
  return static_cast<int>(laneOfThisThread);
}

}

inline void __syncthreads() {
  emu::blockBarrier->arriveAndWait();
}

inline void __syncwarp([[maybe_unused]] unsigned mask = emu::kFullWarpMask) {
  emu::warpOfThisThread->waitForEveryLane();
}

template <class T>
T __shfl_down_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflDownSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_up_sync([[maybe_unused]] unsigned mask, T var, unsigned delta, int width = warpSize) {
  const int sourceLane = emu::shflUpSourceLane(emu::myLane(), static_cast<int>(delta), width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_xor_sync([[maybe_unused]] unsigned mask, T var, int laneMask, int width = warpSize) {
  const int sourceLane = emu::shflXorSourceLane(emu::myLane(), laneMask, width);
  return emu::readValueOfLane(var, sourceLane);
}

template <class T>
T __shfl_sync([[maybe_unused]] unsigned mask, T var, int srcLane, int width = warpSize) {
  const int sourceLane = emu::shflSourceLane(emu::myLane(), srcLane, width);
  return emu::readValueOfLane(var, sourceLane);
}

inline unsigned __ballot_sync([[maybe_unused]] unsigned mask, int predicate) {
  unsigned ballot = 0;
  const unsigned laneCount = emu::warpOfThisThread->laneCount();

  for (unsigned sourceLane = 0; sourceLane < laneCount; ++sourceLane) {
    const int predicateOfLane =
        __shfl_sync(emu::kFullWarpMask, predicate, static_cast<int>(sourceLane));
    if (predicateOfLane) {
      ballot |= 1u << sourceLane;
    }
  }

  return ballot;
}

inline int __any_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) != 0;
}

inline int __all_sync(unsigned mask, int predicate) {
  return __ballot_sync(mask, predicate) == emu::warpOfThisThread->everyLaneMask();
}

namespace emu {

using WarpList = std::vector<std::unique_ptr<Warp>>;

inline unsigned threadCountOf(dim3 block) {
  return block.x * block.y * block.z;
}

inline WarpList splitIntoWarps(unsigned threadCount) {
  WarpList warps;

  for (unsigned firstThread = 0; firstThread < threadCount; firstThread += kWarpSize) {
    const unsigned threadsLeft = threadCount - firstThread;
    const unsigned laneCount = threadsLeft < kWarpSize ? threadsLeft : kWarpSize;
    warps.push_back(std::make_unique<Warp>(laneCount));
  }

  return warps;
}

inline uint3 threadIndexOf(unsigned linearThread, dim3 block) {
  const unsigned x = linearThread % block.x;
  const unsigned y = linearThread / block.x % block.y;
  const unsigned z = linearThread / (block.x * block.y);
  return {x, y, z};
}

inline void waitForTheWholeBlockToFinish() {
  blockBarrier->arriveAndWait();
}

template <class Kernel, class... Args>
void runThreadInEveryBlock(
    unsigned linearThread, Warp* threadWarp, dim3 grid, dim3 block, Kernel kernel, Args... args) {
  warpOfThisThread = threadWarp;
  laneOfThisThread = linearThread % kWarpSize;
  threadIdx = threadIndexOf(linearThread, block);

  for (unsigned bz = 0; bz < grid.z; ++bz) {
    for (unsigned by = 0; by < grid.y; ++by) {
      for (unsigned bx = 0; bx < grid.x; ++bx) {
        blockIdx = {bx, by, bz};
        kernel(args...);
        waitForTheWholeBlockToFinish();
      }
    }
  }
}

constexpr int kErrorInvalidConfiguration = 9;
constexpr unsigned kMaxThreadsPerBlock = 1024;
constexpr unsigned kMaxBlockDimZ = 64;
constexpr unsigned kMaxGridDimYZ = 65535;

inline int lastLaunchError = 0;

inline bool isValidLaunchConfiguration(dim3 grid, dim3 block) {
  const bool blockFits = block.x >= 1 && block.y >= 1 && block.z >= 1 &&
                         block.x <= kMaxThreadsPerBlock && block.y <= kMaxThreadsPerBlock &&
                         block.z <= kMaxBlockDimZ &&
                         threadCountOf(block) <= kMaxThreadsPerBlock;
  const bool gridFits = grid.x >= 1 && grid.y >= 1 && grid.z >= 1 && grid.y <= kMaxGridDimYZ &&
                        grid.z <= kMaxGridDimYZ;
  return blockFits && gridFits;
}

}

template <class Kernel, class... Args>
void launch(Kernel kernel, dim3 grid, dim3 block, Args... args) {
  if (!emu::isValidLaunchConfiguration(grid, block)) {
    emu::lastLaunchError = emu::kErrorInvalidConfiguration;
    return;
  }

  gridDim = grid;
  blockDim = block;

  const unsigned threadCount = emu::threadCountOf(block);
  emu::Barrier barrier(threadCount);
  emu::blockBarrier = &barrier;
  emu::WarpList warps = emu::splitIntoWarps(threadCount);

  std::vector<std::thread> threads;
  threads.reserve(threadCount);
  for (unsigned linearThread = 0; linearThread < threadCount; ++linearThread) {
    emu::Warp* threadWarp = warps[linearThread / emu::kWarpSize].get();
    threads.emplace_back([=] {
      emu::runThreadInEveryBlock(linearThread, threadWarp, grid, block, kernel, args...);
    });
  }

  for (std::thread& thread : threads) {
    thread.join();
  }
  emu::blockBarrier = nullptr;
}

namespace emu {

template <class T, class Improves>
T atomicReplaceWhileImproving(T* address, T candidate, Improves improves) {
  T current = __atomic_load_n(address, kCudaAtomicFunctionOrder);

  while (improves(candidate, current)) {
    const bool replaced = __atomic_compare_exchange_n(
        address, &current, candidate, true, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
    if (replaced) {
      break;
    }
  }

  return current;
}

template <class T>
T atomicStoreMinimum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate < current;
  });
}

template <class T>
T atomicStoreMaximum(T* address, T value) {
  return atomicReplaceWhileImproving(address, value, [](T candidate, T current) {
    return candidate > current;
  });
}

template <class T>
T atomicCompareAndSwap(T* address, T compare, T value) {
  T observed = compare;
  __atomic_compare_exchange_n(
      address, &observed, value, false, kCudaAtomicFunctionOrder, kCudaAtomicFunctionOrder);
  return observed;
}

}

inline int atomicAdd(int* address, int value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned atomicAdd(unsigned* address, unsigned value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline unsigned long long atomicAdd(unsigned long long* address, unsigned long long value) {
  return __atomic_fetch_add(address, value, emu::kCudaAtomicFunctionOrder);
}

inline float atomicAdd(float* address, float value) {
  float old;
  __atomic_load(address, &old, emu::kCudaAtomicFunctionOrder);
  float sum = old + value;

  while (!__atomic_compare_exchange(
      address, &old, &sum, true, emu::kCudaAtomicFunctionOrder, emu::kCudaAtomicFunctionOrder)) {
    sum = old + value;
  }

  return old;
}

inline int atomicMin(int* address, int value) {
  return emu::atomicStoreMinimum(address, value);
}

inline int atomicMax(int* address, int value) {
  return emu::atomicStoreMaximum(address, value);
}

inline unsigned long long atomicMin(unsigned long long* address, unsigned long long value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMin(unsigned* address, unsigned value) {
  return emu::atomicStoreMinimum(address, value);
}

inline unsigned atomicMax(unsigned* address, unsigned value) {
  return emu::atomicStoreMaximum(address, value);
}

inline int atomicSub(int* address, int value) {
  return __atomic_fetch_sub(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicExch(int* address, int value) {
  return __atomic_exchange_n(address, value, emu::kCudaAtomicFunctionOrder);
}

inline int atomicCAS(int* address, int compare, int value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline unsigned long long atomicCAS(unsigned long long* address,
                                    unsigned long long compare,
                                    unsigned long long value) {
  return emu::atomicCompareAndSwap(address, compare, value);
}

inline void __threadfence() {
  __atomic_thread_fence(__ATOMIC_SEQ_CST);
}

namespace cuda {

enum thread_scope {
  thread_scope_system = 0,
  thread_scope_device = 1,
  thread_scope_block = 2,
  thread_scope_thread = 3
};

namespace std {

enum memory_order {
  memory_order_relaxed = __ATOMIC_RELAXED,
  memory_order_consume = __ATOMIC_CONSUME,
  memory_order_acquire = __ATOMIC_ACQUIRE,
  memory_order_release = __ATOMIC_RELEASE,
  memory_order_acq_rel = __ATOMIC_ACQ_REL,
  memory_order_seq_cst = __ATOMIC_SEQ_CST
};

}

using std::memory_order;
using std::memory_order_acq_rel;
using std::memory_order_acquire;
using std::memory_order_relaxed;
using std::memory_order_release;
using std::memory_order_seq_cst;

template <class T, thread_scope Scope = thread_scope_system>
class atomic_ref {
 public:
  explicit atomic_ref(T& object) : address_(&object) {
  }

  T load(memory_order order = memory_order_seq_cst) const {
    T value;
    __atomic_load(address_, &value, order);
    return value;
  }

  void store(T value, memory_order order = memory_order_seq_cst) const {
    __atomic_store(address_, &value, order);
  }

  T exchange(T value, memory_order order = memory_order_seq_cst) const {
    T old;
    __atomic_exchange(address_, &value, &old, order);
    return old;
  }

  bool compare_exchange_strong(T& expected,
                               T desired,
                               memory_order order = memory_order_seq_cst) const {
    return __atomic_compare_exchange(address_, &expected, &desired, false, order, __ATOMIC_RELAXED);
  }

  T fetch_add(T value, memory_order order = memory_order_seq_cst) const {
    if constexpr (::std::is_integral_v<T>) {
      return __atomic_fetch_add(address_, value, order);
    } else {
      return fetchAddByCompareExchange(value, order);
    }
  }

  T fetch_sub(T value, memory_order order = memory_order_seq_cst) const {
    return fetch_add(-value, order);
  }

  T fetch_min(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate < current;
    });
  }

  T fetch_max(T value, memory_order order = memory_order_seq_cst) const {
    return replaceWhileImproving(value, order, [](T candidate, T current) {
      return candidate > current;
    });
  }

 private:
  T fetchAddByCompareExchange(T value, memory_order order) const {
    T old = load(memory_order_relaxed);
    bool replaced = false;
    while (!replaced) {
      replaced = compare_exchange_strong(old, old + value, order);
    }
    return old;
  }

  template <class Improves>
  T replaceWhileImproving(T candidate, memory_order order, Improves improves) const {
    T current = load(memory_order_relaxed);
    while (improves(candidate, current)) {
      if (compare_exchange_strong(current, candidate, order)) {
        break;
      }
    }
    return current;
  }

  T* address_;
};

}

enum cudaMemcpyKind {
  cudaMemcpyHostToDevice,
  cudaMemcpyDeviceToHost,
  cudaMemcpyDeviceToDevice,
  cudaMemcpyHostToHost
};

typedef int cudaError_t;

const cudaError_t cudaSuccess = 0;

const cudaError_t cudaErrorInvalidConfiguration = emu::kErrorInvalidConfiguration;

inline const char* cudaGetErrorString(cudaError_t error) {
  if (error == cudaErrorInvalidConfiguration) {
    return "invalid configuration argument";
  }
  return "no error";
}

inline cudaError_t cudaPeekAtLastError() {
  return emu::lastLaunchError;
}

inline cudaError_t cudaGetLastError() {
  const cudaError_t error = emu::lastLaunchError;
  emu::lastLaunchError = cudaSuccess;
  return error;
}

inline cudaError_t cudaDeviceSynchronize() {
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMalloc(T** devicePointer, size_t bytes) {
  *devicePointer = static_cast<T*>(std::malloc(bytes));
  return cudaSuccess;
}

inline cudaError_t cudaFree(void* devicePointer) {
  std::free(devicePointer);
  return cudaSuccess;
}

inline cudaError_t cudaMemcpy(void* destination, const void* source, size_t bytes, cudaMemcpyKind) {
  std::memcpy(destination, source, bytes);
  return cudaSuccess;
}

inline cudaError_t cudaMemset(void* destination, int byteValue, size_t bytes) {
  std::memset(destination, byteValue, bytes);
  return cudaSuccess;
}

template <class T>
cudaError_t cudaMemcpyToSymbol(T& symbol, const void* source, size_t bytes) {
  std::memcpy(&symbol, source, bytes);
  return cudaSuccess;
}

struct cudaDeviceProp {
  char name[256];
  int major;
  int minor;
  int multiProcessorCount;
  int maxThreadsPerBlock;
  int maxThreadsDim[3];
  int maxGridSize[3];
  int maxThreadsPerMultiProcessor;
  int maxBlocksPerMultiProcessor;
  int warpSize;
  int regsPerBlock;
  int regsPerMultiprocessor;
  size_t sharedMemPerBlock;
  size_t sharedMemPerMultiprocessor;
  size_t totalGlobalMem;
  size_t totalConstMem;
  int clockRate;
};

inline cudaError_t cudaGetDeviceCount(int* deviceCount) {
  *deviceCount = 1;
  return cudaSuccess;
}

inline cudaError_t cudaGetDeviceProperties(cudaDeviceProp* properties, int) {
  constexpr int MAX_THREADS_PER_BLOCK = 1024;
  constexpr int MAX_GRID_X = 2147483647;
  constexpr int MAX_GRID_YZ = 65535;
  constexpr int MAX_BLOCK_Z = 64;
  constexpr size_t KIB = 1024;
  *properties = cudaDeviceProp{};
  std::snprintf(properties->name, sizeof properties->name, "CPU emulation (cuda_compat.h)");
  properties->multiProcessorCount = static_cast<int>(std::thread::hardware_concurrency());
  properties->maxThreadsPerBlock = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[0] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[1] = MAX_THREADS_PER_BLOCK;
  properties->maxThreadsDim[2] = MAX_BLOCK_Z;
  properties->maxGridSize[0] = MAX_GRID_X;
  properties->maxGridSize[1] = MAX_GRID_YZ;
  properties->maxGridSize[2] = MAX_GRID_YZ;
  properties->maxThreadsPerMultiProcessor = 2 * MAX_THREADS_PER_BLOCK;
  properties->maxBlocksPerMultiProcessor = 32;
  properties->warpSize = warpSize;
  properties->regsPerBlock = 64 * 1024;
  properties->regsPerMultiprocessor = 64 * 1024;
  properties->sharedMemPerBlock = 48 * KIB;
  properties->sharedMemPerMultiprocessor = 100 * KIB;
  properties->totalConstMem = 64 * KIB;
  return cudaSuccess;
}

#endif

#define CHECK(call)                                                                          \
  do {                                                                                       \
    cudaError_t err_ = (call);                                                               \
    if (err_ != cudaSuccess) {                                                               \
      std::fprintf(                                                                          \
          stderr, "CUDA error %s at %s:%d\n", cudaGetErrorString(err_), __FILE__, __LINE__); \
      std::exit(1);                                                                          \
    }                                                                                        \
  } while (0)

struct Timer {
  std::chrono::steady_clock::time_point t0 = std::chrono::steady_clock::now();

  double ms() {
    cudaDeviceSynchronize();
    const auto elapsed = std::chrono::steady_clock::now() - t0;
    return std::chrono::duration<double, std::milli>(elapsed).count();
  }
};

inline unsigned cdiv(unsigned a, unsigned b) {
  return (a + b - 1) / b;
}

__host__ __device__ inline int imin(int a, int b) {
  return a < b ? a : b;
}

__host__ __device__ inline int imax(int a, int b) {
  return a > b ? a : b;
}

__host__ __device__ inline float fminf_(float a, float b) {
  return a < b ? a : b;
}

__host__ __device__ inline float fmaxf_(float a, float b) {
  return a > b ? a : b;
}


## Step 1 of 5: `01_merge_sequential.cu`

The sequential merge of two small sorted arrays, run by a single GPU thread. Ties go to `A` (`<=`), so the merge is stable.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 01_merge_sequential.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdio>
#include <vector>

constexpr int M = 5;
constexpr int N = 6;

__host__ __device__ void merge_sequential(const int* A, int m, const int* B, int n, int* C) {
  int i = 0;
  int j = 0;
  int k = 0;
  while (i < m && j < n) {
    if (A[i] <= B[j]) {
      C[k++] = A[i++];
    } else {
      C[k++] = B[j++];
    }
  }
  while (i < m) {
    C[k++] = A[i++];
  }
  while (j < n) {
    C[k++] = B[j++];
  }
}

__global__ void merge_sequential_kernel(const int* A, int m, const int* B, int n, int* C) {
  merge_sequential(A, m, B, n, C);
}

static void printArray(const char* label, const std::vector<int>& values) {
  std::printf("%s", label);
  for (int value : values) {
    std::printf(" %3d", value);
  }
  std::printf("\n");
}

int main() {
  const std::vector<int> A = {3, 5, 5, 8, 12};
  const std::vector<int> B = {1, 5, 6, 8, 9, 14};
  std::vector<int> C(M + N);

  int* A_d = nullptr;
  int* B_d = nullptr;
  int* C_d = nullptr;
  CHECK(cudaMalloc(&A_d, M * sizeof(int)));
  CHECK(cudaMalloc(&B_d, N * sizeof(int)));
  CHECK(cudaMalloc(&C_d, (M + N) * sizeof(int)));
  CHECK(cudaMemcpy(A_d, A.data(), M * sizeof(int), cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(B_d, B.data(), N * sizeof(int), cudaMemcpyHostToDevice));

  launch(merge_sequential_kernel, dim3(1), dim3(1), A_d, M, B_d, N, C_d);
  CHECK(cudaGetLastError());
  CHECK(cudaMemcpy(C.data(), C_d, (M + N) * sizeof(int), cudaMemcpyDeviceToHost));

  std::vector<int> expected(M + N);
  std::merge(A.begin(), A.end(), B.begin(), B.end(), expected.begin());

  printArray("A:", A);
  printArray("B:", B);
  printArray("C:", C);

  CHECK(cudaFree(A_d));
  CHECK(cudaFree(B_d));
  CHECK(cudaFree(C_d));

  const bool ok = C == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 01_merge_sequential.cu -o 01_merge_sequential && ./01_merge_sequential

**What to look at in the output:** C has 11 values, 5 from A and 6 from B. Look at the three 5s: the two 5s of A (C[2] and C[3]) come before the 5 of B (C[4]), and the 8 of A comes before the 8 of B. With plain integers the order of equal values is invisible, but with records it is what makes the merge stable.

## Step 2 of 5: `02_co_rank.cu`

The co-rank function: one thread per output position `k` finds how many elements of `A` the first `k` outputs take. The binary search for `k = 6` is printed step by step, as in the worked example of the README.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 02_co_rank.cu
#include "cuda_compat.h"

#include <cstdio>
#include <vector>

constexpr int M = 5;
constexpr int N = 6;
constexpr int TRACED_K = 6;

__host__ __device__ int co_rank(int k, const int* A, int m, const int* B, int n) {
  int i = imin(k, m);
  int j = k - i;
  int i_low = imax(0, k - n);
  int j_low = imax(0, k - m);
  bool searching = true;
  while (searching) {
    const bool i_too_large = i > 0 && j < n && A[i - 1] > B[j];
    const bool i_too_small = j > 0 && i < m && B[j - 1] >= A[i];
    if (i_too_large) {
      const int delta = (i - i_low + 1) >> 1;
      j_low = j;
      i -= delta;
      j += delta;
    } else if (i_too_small) {
      const int delta = (j - j_low + 1) >> 1;
      i_low = i;
      i += delta;
      j -= delta;
    } else {
      searching = false;
    }
  }
  return i;
}

__global__ void co_rank_kernel(const int* A, int m, const int* B, int n, int* coRanks) {
  const int k = blockIdx.x * blockDim.x + threadIdx.x;
  if (k <= m + n) {
    coRanks[k] = co_rank(k, A, m, B, n);
  }
}

static void traceCoRank(int k, const int* A, int m, const int* B, int n) {
  int i = imin(k, m);
  int j = k - i;
  int i_low = imax(0, k - n);
  int j_low = imax(0, k - m);
  std::printf("co_rank search for k = %d\n", k);
  bool searching = true;
  while (searching) {
    std::printf("  i = %d  j = %d  i_low = %d  j_low = %d  ", i, j, i_low, j_low);
    const bool i_too_large = i > 0 && j < n && A[i - 1] > B[j];
    const bool i_too_small = j > 0 && i < m && B[j - 1] >= A[i];
    if (i_too_large) {
      std::printf("A[%d]=%d > B[%d]=%d: i too large\n", i - 1, A[i - 1], j, B[j]);
      const int delta = (i - i_low + 1) >> 1;
      j_low = j;
      i -= delta;
      j += delta;
    } else if (i_too_small) {
      std::printf("B[%d]=%d >= A[%d]=%d: i too small\n", j - 1, B[j - 1], i, A[i]);
      const int delta = (j - j_low + 1) >> 1;
      i_low = i;
      i += delta;
      j -= delta;
    } else {
      std::printf("both conditions hold: stop\n");
      searching = false;
    }
  }
}

static void printArray(const char* label, const std::vector<int>& values) {
  std::printf("%s", label);
  for (int value : values) {
    std::printf(" %3d", value);
  }
  std::printf("\n");
}

static std::vector<int> coRanksOnHost(const std::vector<int>& A, const std::vector<int>& B) {
  std::vector<int> coRanks = {0};
  int i = 0;
  int j = 0;
  while (i < M || j < N) {
    const bool takeA = j == N || (i < M && A[i] <= B[j]);
    i += takeA ? 1 : 0;
    j += takeA ? 0 : 1;
    coRanks.push_back(i);
  }
  return coRanks;
}

int main() {
  const std::vector<int> A = {3, 5, 5, 8, 12};
  const std::vector<int> B = {1, 5, 6, 8, 9, 14};
  const int outputs = M + N;
  std::vector<int> coRanks(outputs + 1);

  int* A_d = nullptr;
  int* B_d = nullptr;
  int* coRanks_d = nullptr;
  CHECK(cudaMalloc(&A_d, M * sizeof(int)));
  CHECK(cudaMalloc(&B_d, N * sizeof(int)));
  CHECK(cudaMalloc(&coRanks_d, (outputs + 1) * sizeof(int)));
  CHECK(cudaMemcpy(A_d, A.data(), M * sizeof(int), cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(B_d, B.data(), N * sizeof(int), cudaMemcpyHostToDevice));

  launch(co_rank_kernel, dim3(1), dim3(outputs + 1), A_d, M, B_d, N, coRanks_d);
  CHECK(cudaGetLastError());
  CHECK(cudaMemcpy(coRanks.data(), coRanks_d, (outputs + 1) * sizeof(int), cudaMemcpyDeviceToHost));

  printArray("A:", A);
  printArray("B:", B);
  std::printf("\n");
  traceCoRank(TRACED_K, A.data(), M, B.data(), N);
  std::printf("\n k  i = co_rank(k)  j = k - i\n");
  for (int k = 0; k <= outputs; ++k) {
    std::printf("%2d  %14d  %9d\n", k, coRanks[k], k - coRanks[k]);
  }

  CHECK(cudaFree(A_d));
  CHECK(cudaFree(B_d));
  CHECK(cudaFree(coRanks_d));

  const bool ok = coRanks == coRanksOnHost(A, B);
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 02_co_rank.cu -o 02_co_rank && ./02_co_rank

**What to look at in the output:** The trace starts at i = 5, finds `A[4]=12 > B[1]=5` (too many elements from A), halves the range and stops at i = 3, j = 3: the first 6 outputs take A[0, 3) and B[0, 3). In the table i grows by 1 when output k came from A and j when it came from B, for example from k = 4 to k = 5 only j grows, because C[4] is the 5 of B. Try: set `TRACED_K = 9`; the search stops after two lines at i = 4, j = 5, the same as the table row for k = 9.

## Step 3 of 5: `03_merge_basic.cu`

The basic parallel merge kernel: each of 4 threads owns a fixed range of outputs, finds its input ranges with two co-rank calls and merges them sequentially. The program prints the range of every thread.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 03_merge_basic.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdio>
#include <vector>

constexpr int M = 5;
constexpr int N = 6;
constexpr int THREADS = 4;

__host__ __device__ void merge_sequential(const int* A, int m, const int* B, int n, int* C) {
  int i = 0;
  int j = 0;
  int k = 0;
  while (i < m && j < n) {
    if (A[i] <= B[j]) {
      C[k++] = A[i++];
    } else {
      C[k++] = B[j++];
    }
  }
  while (i < m) {
    C[k++] = A[i++];
  }
  while (j < n) {
    C[k++] = B[j++];
  }
}

__host__ __device__ int co_rank(int k, const int* A, int m, const int* B, int n) {
  int i = imin(k, m);
  int j = k - i;
  int i_low = imax(0, k - n);
  int j_low = imax(0, k - m);
  bool searching = true;
  while (searching) {
    const bool i_too_large = i > 0 && j < n && A[i - 1] > B[j];
    const bool i_too_small = j > 0 && i < m && B[j - 1] >= A[i];
    if (i_too_large) {
      const int delta = (i - i_low + 1) >> 1;
      j_low = j;
      i -= delta;
      j += delta;
    } else if (i_too_small) {
      const int delta = (j - j_low + 1) >> 1;
      i_low = i;
      i += delta;
      j -= delta;
    } else {
      searching = false;
    }
  }
  return i;
}

__global__ void merge_basic_kernel(const int* A, int m, const int* B, int n, int* C) {
  const int tid = blockIdx.x * blockDim.x + threadIdx.x;
  const int totalThreads = gridDim.x * blockDim.x;
  const int elementsPerThread = (m + n + totalThreads - 1) / totalThreads;

  const int k_curr = imin(tid * elementsPerThread, m + n);
  const int k_next = imin(k_curr + elementsPerThread, m + n);
  const int i_curr = co_rank(k_curr, A, m, B, n);
  const int i_next = co_rank(k_next, A, m, B, n);
  const int j_curr = k_curr - i_curr;
  const int j_next = k_next - i_next;

  merge_sequential(A + i_curr, i_next - i_curr, B + j_curr, j_next - j_curr, C + k_curr);
}

static void printArray(const char* label, const std::vector<int>& values) {
  std::printf("%s", label);
  for (int value : values) {
    std::printf(" %3d", value);
  }
  std::printf("\n");
}

static void printThreadPieces(const std::vector<int>& A, const std::vector<int>& B) {
  const int elementsPerThread = (M + N + THREADS - 1) / THREADS;
  for (int tid = 0; tid < THREADS; ++tid) {
    const int k_curr = imin(tid * elementsPerThread, M + N);
    const int k_next = imin(k_curr + elementsPerThread, M + N);
    const int i_curr = co_rank(k_curr, A.data(), M, B.data(), N);
    const int i_next = co_rank(k_next, A.data(), M, B.data(), N);
    std::printf("thread %d: C[%2d, %2d) = merge(A[%d, %d), B[%d, %d))\n",
                tid,
                k_curr,
                k_next,
                i_curr,
                i_next,
                k_curr - i_curr,
                k_next - i_next);
  }
}

int main() {
  const std::vector<int> A = {3, 5, 5, 8, 12};
  const std::vector<int> B = {1, 5, 6, 8, 9, 14};
  std::vector<int> C(M + N);

  int* A_d = nullptr;
  int* B_d = nullptr;
  int* C_d = nullptr;
  CHECK(cudaMalloc(&A_d, M * sizeof(int)));
  CHECK(cudaMalloc(&B_d, N * sizeof(int)));
  CHECK(cudaMalloc(&C_d, (M + N) * sizeof(int)));
  CHECK(cudaMemcpy(A_d, A.data(), M * sizeof(int), cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(B_d, B.data(), N * sizeof(int), cudaMemcpyHostToDevice));

  launch(merge_basic_kernel, dim3(1), dim3(THREADS), A_d, M, B_d, N, C_d);
  CHECK(cudaGetLastError());
  CHECK(cudaMemcpy(C.data(), C_d, (M + N) * sizeof(int), cudaMemcpyDeviceToHost));

  std::vector<int> expected(M + N);
  std::merge(A.begin(), A.end(), B.begin(), B.end(), expected.begin());

  printArray("A:", A);
  printArray("B:", B);
  printThreadPieces(A, B);
  printArray("C:", C);

  CHECK(cudaFree(A_d));
  CHECK(cudaFree(B_d));
  CHECK(cudaFree(C_d));

  const bool ok = C == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 03_merge_basic.cu -o 03_merge_basic && ./03_merge_basic

**What to look at in the output:** Every thread owns 3 outputs (the last one 2), and its line gives the input ranges its two co-rank calls found, for example thread 1 merges A[2, 3) = 5 with B[1, 3) = 5 6. The input ranges have different lengths (thread 0 takes 2 elements of A and 1 of B), but the output ranges are fixed in advance. Try: set `THREADS = 3`; each thread owns 4 outputs and thread 0 takes A[0, 3) and B[0, 1).

## Step 4 of 5: `04_merge_tiled.cu`

The tiled kernel: each block finds its ranges once, then loads `TILE` elements of `A` and of `B` into `A_S` and `B_S` with coalesced loads and runs the co-rank inside shared memory (2 blocks, 2 threads each, `TILE = 4`).

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 04_merge_tiled.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdio>
#include <vector>

constexpr int TILE = 4;
constexpr int TILED_THREADS_PER_BLOCK = 2;
constexpr int TILE_OUTPUTS_PER_THREAD = TILE / TILED_THREADS_PER_BLOCK;
constexpr int BLOCKS = 2;
static_assert(TILE % TILED_THREADS_PER_BLOCK == 0, "every thread merges the same share of a tile");

__host__ __device__ void merge_sequential(const int* A, int m, const int* B, int n, int* C) {
  int i = 0;
  int j = 0;
  int k = 0;
  while (i < m && j < n) {
    if (A[i] <= B[j]) {
      C[k++] = A[i++];
    } else {
      C[k++] = B[j++];
    }
  }
  while (i < m) {
    C[k++] = A[i++];
  }
  while (j < n) {
    C[k++] = B[j++];
  }
}

__host__ __device__ int co_rank(int k, const int* A, int m, const int* B, int n) {
  int i = imin(k, m);
  int j = k - i;
  int i_low = imax(0, k - n);
  int j_low = imax(0, k - m);
  bool searching = true;
  while (searching) {
    const bool i_too_large = i > 0 && j < n && A[i - 1] > B[j];
    const bool i_too_small = j > 0 && i < m && B[j - 1] >= A[i];
    if (i_too_large) {
      const int delta = (i - i_low + 1) >> 1;
      j_low = j;
      i -= delta;
      j += delta;
    } else if (i_too_small) {
      const int delta = (j - j_low + 1) >> 1;
      i_low = i;
      i += delta;
      j -= delta;
    } else {
      searching = false;
    }
  }
  return i;
}

__global__ void merge_tiled_kernel(const int* A, int m, const int* B, int n, int* C) {
  __shared__ int A_S[TILE];
  __shared__ int B_S[TILE];
  __shared__ int block_A_curr;
  __shared__ int block_A_next;

  const int outputsPerBlock = (m + n + gridDim.x - 1) / gridDim.x;
  const int C_curr = imin(blockIdx.x * outputsPerBlock, m + n);
  const int C_next = imin(C_curr + outputsPerBlock, m + n);
  if (threadIdx.x == 0) {
    block_A_curr = co_rank(C_curr, A, m, B, n);
    block_A_next = co_rank(C_next, A, m, B, n);
  }
  __syncthreads();

  const int A_curr = block_A_curr;
  const int A_next = block_A_next;
  const int B_curr = C_curr - A_curr;
  const int B_next = C_next - A_next;
  const int A_length = A_next - A_curr;
  const int B_length = B_next - B_curr;
  const int C_length = C_next - C_curr;

  int C_completed = 0;
  int A_consumed = 0;
  int B_consumed = 0;
  while (C_completed < C_length) {
    const int A_S_length = imin(TILE, A_length - A_consumed);
    const int B_S_length = imin(TILE, B_length - B_consumed);
    for (int x = threadIdx.x; x < A_S_length; x += blockDim.x) {
      A_S[x] = A[A_curr + A_consumed + x];
    }
    for (int x = threadIdx.x; x < B_S_length; x += blockDim.x) {
      B_S[x] = B[B_curr + B_consumed + x];
    }
    __syncthreads();

    const int C_step = imin(TILE, C_length - C_completed);
    const int c_curr = imin(threadIdx.x * TILE_OUTPUTS_PER_THREAD, C_step);
    const int c_next = imin(c_curr + TILE_OUTPUTS_PER_THREAD, C_step);
    const int a_curr = co_rank(c_curr, A_S, A_S_length, B_S, B_S_length);
    const int a_next = co_rank(c_next, A_S, A_S_length, B_S, B_S_length);
    const int b_curr = c_curr - a_curr;
    const int b_next = c_next - a_next;
    merge_sequential(A_S + a_curr,
                     a_next - a_curr,
                     B_S + b_curr,
                     b_next - b_curr,
                     C + C_curr + C_completed + c_curr);

    const int A_step = co_rank(C_step, A_S, A_S_length, B_S, B_S_length);
    A_consumed += A_step;
    B_consumed += C_step - A_step;
    C_completed += C_step;
    __syncthreads();
  }
}

static void printArray(const char* label, const std::vector<int>& values) {
  std::printf("%s", label);
  for (int value : values) {
    std::printf(" %3d", value);
  }
  std::printf("\n");
}

static void printBlockRanges(const std::vector<int>& A, const std::vector<int>& B) {
  const int m = A.size();
  const int n = B.size();
  const int outputsPerBlock = (m + n + BLOCKS - 1) / BLOCKS;
  for (int block = 0; block < BLOCKS; ++block) {
    const int C_curr = imin(block * outputsPerBlock, m + n);
    const int C_next = imin(C_curr + outputsPerBlock, m + n);
    const int A_curr = co_rank(C_curr, A.data(), m, B.data(), n);
    const int A_next = co_rank(C_next, A.data(), m, B.data(), n);
    std::printf("block %d: C[%2d, %2d) = merge(A[%2d, %2d), B[%2d, %2d)) in tiles of %d\n",
                block,
                C_curr,
                C_next,
                A_curr,
                A_next,
                C_curr - A_curr,
                C_next - A_next,
                TILE);
  }
}

int main() {
  const std::vector<int> A = {1, 3, 4, 4, 7, 9, 10, 12, 15, 18};
  const std::vector<int> B = {2, 4, 5, 6, 8, 11, 11, 13, 14, 16, 17, 20};
  const int m = A.size();
  const int n = B.size();
  std::vector<int> C(m + n);

  int* A_d = nullptr;
  int* B_d = nullptr;
  int* C_d = nullptr;
  CHECK(cudaMalloc(&A_d, m * sizeof(int)));
  CHECK(cudaMalloc(&B_d, n * sizeof(int)));
  CHECK(cudaMalloc(&C_d, (m + n) * sizeof(int)));
  CHECK(cudaMemcpy(A_d, A.data(), m * sizeof(int), cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(B_d, B.data(), n * sizeof(int), cudaMemcpyHostToDevice));

  launch(merge_tiled_kernel, dim3(BLOCKS), dim3(TILED_THREADS_PER_BLOCK), A_d, m, B_d, n, C_d);
  CHECK(cudaGetLastError());
  CHECK(cudaMemcpy(C.data(), C_d, (m + n) * sizeof(int), cudaMemcpyDeviceToHost));

  std::vector<int> expected(m + n);
  std::merge(A.begin(), A.end(), B.begin(), B.end(), expected.begin());

  printArray("A:", A);
  printArray("B:", B);
  printBlockRanges(A, B);
  printArray("C:", C);

  CHECK(cudaFree(A_d));
  CHECK(cudaFree(B_d));
  CHECK(cudaFree(C_d));

  const bool ok = C == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 04_merge_tiled.cu -o 04_merge_tiled && ./04_merge_tiled

**What to look at in the output:** Each block owns 11 of the 22 outputs: block 0 merges A[0, 6) with B[0, 5) and block 1 merges A[6, 10) with B[5, 12), so block 1 takes 7 elements of B and only 4 of A. With `TILE = 4` each block works in three rounds of 4, 4 and 3 outputs, loading up to 4 elements of each input into shared memory per round. Try: set `BLOCKS = 3`; each block owns 8 outputs and block 2 merges A[8, 10) with B[8, 12).

## Step 5 of 5: `05_merge_circular.cu`

Circular buffers: the unused part of each tile stays in shared memory and only the consumed slots are refilled, so every input element is loaded once. Every index into a tile goes through `circular_slot`.

Run the next cell to save the program, then the one after it to compile and run it.

In [ ]:
%%writefile 05_merge_circular.cu
#include "cuda_compat.h"

#include <algorithm>
#include <cstdio>
#include <vector>

constexpr int TILE = 4;
constexpr int TILED_THREADS_PER_BLOCK = 2;
constexpr int TILE_OUTPUTS_PER_THREAD = TILE / TILED_THREADS_PER_BLOCK;
constexpr int BLOCKS = 2;
constexpr int TILE_MASK = TILE - 1;
static_assert(TILE % TILED_THREADS_PER_BLOCK == 0, "every thread merges the same share of a tile");
static_assert((TILE & TILE_MASK) == 0, "TILE must be a power of two");

__host__ __device__ int co_rank(int k, const int* A, int m, const int* B, int n) {
  int i = imin(k, m);
  int j = k - i;
  int i_low = imax(0, k - n);
  int j_low = imax(0, k - m);
  bool searching = true;
  while (searching) {
    const bool i_too_large = i > 0 && j < n && A[i - 1] > B[j];
    const bool i_too_small = j > 0 && i < m && B[j - 1] >= A[i];
    if (i_too_large) {
      const int delta = (i - i_low + 1) >> 1;
      j_low = j;
      i -= delta;
      j += delta;
    } else if (i_too_small) {
      const int delta = (j - j_low + 1) >> 1;
      i_low = i;
      i += delta;
      j -= delta;
    } else {
      searching = false;
    }
  }
  return i;
}

__host__ __device__ int circular_slot(int index) {
  return index & TILE_MASK;
}

__host__ __device__ int co_rank_circular(
    int k, const int* A_S, int A_S_start, int m, const int* B_S, int B_S_start, int n) {
  int i = imin(k, m);
  int j = k - i;
  int i_low = imax(0, k - n);
  int j_low = imax(0, k - m);
  bool searching = true;
  while (searching) {
    const bool i_too_large =
        i > 0 && j < n && A_S[circular_slot(A_S_start + i - 1)] > B_S[circular_slot(B_S_start + j)];
    const bool i_too_small =
        j > 0 && i < m &&
        B_S[circular_slot(B_S_start + j - 1)] >= A_S[circular_slot(A_S_start + i)];
    if (i_too_large) {
      const int delta = (i - i_low + 1) >> 1;
      j_low = j;
      i -= delta;
      j += delta;
    } else if (i_too_small) {
      const int delta = (j - j_low + 1) >> 1;
      i_low = i;
      i += delta;
      j -= delta;
    } else {
      searching = false;
    }
  }
  return i;
}

__host__ __device__ void merge_sequential_circular(
    const int* A_S, int A_S_start, int m, const int* B_S, int B_S_start, int n, int* C) {
  int i = 0;
  int j = 0;
  int k = 0;
  while (i < m && j < n) {
    const int a = A_S[circular_slot(A_S_start + i)];
    const int b = B_S[circular_slot(B_S_start + j)];
    if (a <= b) {
      C[k++] = a;
      ++i;
    } else {
      C[k++] = b;
      ++j;
    }
  }
  while (i < m) {
    C[k++] = A_S[circular_slot(A_S_start + i++)];
  }
  while (j < n) {
    C[k++] = B_S[circular_slot(B_S_start + j++)];
  }
}

__global__ void merge_circular_kernel(const int* A, int m, const int* B, int n, int* C) {
  __shared__ int A_S[TILE];
  __shared__ int B_S[TILE];
  __shared__ int block_A_curr;
  __shared__ int block_A_next;

  const int outputsPerBlock = (m + n + gridDim.x - 1) / gridDim.x;
  const int C_curr = imin(blockIdx.x * outputsPerBlock, m + n);
  const int C_next = imin(C_curr + outputsPerBlock, m + n);
  if (threadIdx.x == 0) {
    block_A_curr = co_rank(C_curr, A, m, B, n);
    block_A_next = co_rank(C_next, A, m, B, n);
  }
  __syncthreads();

  const int A_curr = block_A_curr;
  const int A_next = block_A_next;
  const int B_curr = C_curr - A_curr;
  const int B_next = C_next - A_next;
  const int A_length = A_next - A_curr;
  const int B_length = B_next - B_curr;
  const int C_length = C_next - C_curr;

  int C_completed = 0;
  int A_consumed = 0;
  int B_consumed = 0;
  int A_loaded = 0;
  int B_loaded = 0;
  while (C_completed < C_length) {
    const int A_window_end = imin(A_consumed + TILE, A_length);
    const int B_window_end = imin(B_consumed + TILE, B_length);
    for (int x = A_loaded + threadIdx.x; x < A_window_end; x += blockDim.x) {
      A_S[circular_slot(x)] = A[A_curr + x];
    }
    for (int x = B_loaded + threadIdx.x; x < B_window_end; x += blockDim.x) {
      B_S[circular_slot(x)] = B[B_curr + x];
    }
    A_loaded = A_window_end;
    B_loaded = B_window_end;
    __syncthreads();

    const int A_S_length = A_loaded - A_consumed;
    const int B_S_length = B_loaded - B_consumed;
    const int C_step = imin(TILE, C_length - C_completed);
    const int c_curr = imin(threadIdx.x * TILE_OUTPUTS_PER_THREAD, C_step);
    const int c_next = imin(c_curr + TILE_OUTPUTS_PER_THREAD, C_step);
    const int a_curr =
        co_rank_circular(c_curr, A_S, A_consumed, A_S_length, B_S, B_consumed, B_S_length);
    const int a_next =
        co_rank_circular(c_next, A_S, A_consumed, A_S_length, B_S, B_consumed, B_S_length);
    const int b_curr = c_curr - a_curr;
    const int b_next = c_next - a_next;
    merge_sequential_circular(A_S,
                              A_consumed + a_curr,
                              a_next - a_curr,
                              B_S,
                              B_consumed + b_curr,
                              b_next - b_curr,
                              C + C_curr + C_completed + c_curr);

    const int A_step =
        co_rank_circular(C_step, A_S, A_consumed, A_S_length, B_S, B_consumed, B_S_length);
    A_consumed += A_step;
    B_consumed += C_step - A_step;
    C_completed += C_step;
    __syncthreads();
  }
}

static void printArray(const char* label, const std::vector<int>& values) {
  std::printf("%s", label);
  for (int value : values) {
    std::printf(" %3d", value);
  }
  std::printf("\n");
}

int main() {
  const std::vector<int> A = {1, 3, 4, 4, 7, 9, 10, 12, 15, 18};
  const std::vector<int> B = {2, 4, 5, 6, 8, 11, 11, 13, 14, 16, 17, 20};
  const int m = A.size();
  const int n = B.size();
  std::vector<int> C(m + n);

  int* A_d = nullptr;
  int* B_d = nullptr;
  int* C_d = nullptr;
  CHECK(cudaMalloc(&A_d, m * sizeof(int)));
  CHECK(cudaMalloc(&B_d, n * sizeof(int)));
  CHECK(cudaMalloc(&C_d, (m + n) * sizeof(int)));
  CHECK(cudaMemcpy(A_d, A.data(), m * sizeof(int), cudaMemcpyHostToDevice));
  CHECK(cudaMemcpy(B_d, B.data(), n * sizeof(int), cudaMemcpyHostToDevice));

  launch(merge_circular_kernel, dim3(BLOCKS), dim3(TILED_THREADS_PER_BLOCK), A_d, m, B_d, n, C_d);
  CHECK(cudaGetLastError());
  CHECK(cudaMemcpy(C.data(), C_d, (m + n) * sizeof(int), cudaMemcpyDeviceToHost));

  std::vector<int> expected(m + n);
  std::merge(A.begin(), A.end(), B.begin(), B.end(), expected.begin());

  printArray("A:", A);
  printArray("B:", B);
  printArray("C:", C);

  CHECK(cudaFree(A_d));
  CHECK(cudaFree(B_d));
  CHECK(cudaFree(C_d));

  const bool ok = C == expected;
  std::printf("%s\n", ok ? "PASS" : "FAIL");
  return ok ? 0 : 1;
}


In [ ]:
!nvcc -O2 -arch=native -I. 05_merge_circular.cu -o 05_merge_circular && ./05_merge_circular

**What to look at in the output:** The output is the same C as in step 4, and nothing about the tiles is printed: the difference is in the loads. The tiled kernel reloads a full tile every round, even the elements it did not consume, while here only the consumed slots are refilled, at `circular_slot(x) = x & 3`, so each input element is read from global memory once.

## What you did

Every program above was checked against the CPU. Go back to any `%%writefile` cell, change something (a size, a block dimension, a constant) and run the pair again to see what changes.

The exercises of this topic, with solutions, are at the end of `ec2/11_merge/README.md`.

**Next:** open `colab_2_full_programs.ipynb` in this same folder.